In [1]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import polars as pl
import random

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

In [3]:
# Collect all home_team parquet files from extracted data folder

home_team_files = sorted(
    [
        file
        for file in EXTRACT_ROOT.glob("*/raw_match_parquet/home_team_*.parquet")\
        if "home_team_score" not in file.name
    ]
)

print("Number of away team files:", len(home_team_files))

Number of away team files: 25610


In [4]:
# Group parquet files based on match_id
# Files with the same match_id may represent different snapshots of the same match

match_files = defaultdict(list)


for file in home_team_files:
    match_id = file.stem.replace("home_team_", "")
    match_files[match_id].append(file)


print("Total unique match IDs:", len(match_files))

Total unique match IDs: 12389


In [5]:
# Find match IDs that have more than one parquet file
# Multiple files may indicate multiple snapshots collected at different times

duplicate_matches = {
    match_id: files
    for match_id, files in match_files.items()
    if len(files) > 1
}


print(
    "Number of match IDs with multiple snapshots:",
    len(duplicate_matches)
)

Number of match IDs with multiple snapshots: 11587


In [6]:
# Display number of snapshots available for some duplicated match IDs

for match_id, files in list(duplicate_matches.items())[:10]:
    
    print(
        "Match ID:",
        match_id,
        "Number of snapshots:",
        len(files)
    )

Match ID: 11998445 Number of snapshots: 2
Match ID: 11998446 Number of snapshots: 2
Match ID: 11998447 Number of snapshots: 2
Match ID: 11998456 Number of snapshots: 3
Match ID: 11998459 Number of snapshots: 2
Match ID: 11998666 Number of snapshots: 2
Match ID: 11998667 Number of snapshots: 2
Match ID: 11998670 Number of snapshots: 2
Match ID: 11998671 Number of snapshots: 2
Match ID: 11998772 Number of snapshots: 2


In [7]:
# Randomly select match IDs to investigate snapshot differences

sample_match_ids = random.sample(
    list(duplicate_matches.keys()),
    5
)


print(sample_match_ids)

['12166343', '12145023', '12049583', '12110075', '12150630']


In [8]:
# Compare snapshots of one match ID
# This code identifies columns that changed between snapshots

match_id = sample_match_ids[0]


files = duplicate_matches[match_id]


print("Analyzing Match ID:", match_id)
print("Number of snapshots:", len(files))


# Read all snapshots of this match

dfs = [
    pl.read_parquet(file)
    for file in files
]


# Compare every column

for column in dfs[0].columns:
    
    unique_values = set()

    for df in dfs:
        
        values = df[column].unique().to_list()
        
        unique_values.update(values)


    # If more than one value exists, the column changed between snapshots
    
    if len(unique_values) > 1:
        
        print(
            "Changed column:",
            column
        )

        print(
            "Values:",
            unique_values
        )

        print("----------------------")

Analyzing Match ID: 12166343
Number of snapshots: 2
Changed column: user_count
Values: {1482, 1476}
----------------------


In [9]:
# Compare multiple random match IDs automatically

for match_id in sample_match_ids:

    print("==============================")
    print("Match ID:", match_id)


    files = duplicate_matches[match_id]


    dfs = [
        pl.read_parquet(file)
        for file in files
    ]


    changed_columns = []


    for column in dfs[0].columns:

        values = set()

        for df in dfs:
            values.update(
                df[column].unique().to_list()
            )


        if len(values) > 1:
            changed_columns.append(column)


    print(
        "Changed columns:",
        changed_columns
    )

Match ID: 12166343
Changed columns: ['user_count']
Match ID: 12145023
Changed columns: []
Match ID: 12049583
Changed columns: []
Match ID: 12110075
Changed columns: []
Match ID: 12150630
Changed columns: ['user_count', 'height', 'current_prize', 'total_prize', 'current_rank']


In [10]:
# Find match IDs with the highest number of snapshots
# Each file with the same match_id represents one snapshot

snapshot_counts = {
    match_id: len(files)
    for match_id, files in match_files.items()
}


# Sort match IDs by number of snapshots (descending order)

top_snapshot_matches = sorted(
    snapshot_counts.items(),
    key=lambda x: x[1],
    reverse=True
)


# Display top 10 match IDs with the most snapshots

print("Top 10 match IDs with the highest number of snapshots")
print("=" * 50)


for match_id, count in top_snapshot_matches[:10]:

    print(
        f"Match ID: {match_id} | Number of snapshots: {count}"
    )

Top 10 match IDs with the highest number of snapshots
Match ID: 12063582 | Number of snapshots: 4
Match ID: 12063583 | Number of snapshots: 4
Match ID: 12063587 | Number of snapshots: 4
Match ID: 12063588 | Number of snapshots: 4
Match ID: 12063611 | Number of snapshots: 4
Match ID: 12063615 | Number of snapshots: 4
Match ID: 12084420 | Number of snapshots: 4
Match ID: 12086016 | Number of snapshots: 4
Match ID: 12195781 | Number of snapshots: 4
Match ID: 11998456 | Number of snapshots: 3


In [11]:
# Count distribution of number of snapshots per match ID
# Shows how many matches have 1, 2, 3, ... snapshots

from collections import Counter


snapshot_distribution = Counter(
    len(files)
    for files in match_files.values()
)


for snapshots, count in sorted(snapshot_distribution.items()):

    print(
        f"{snapshots} snapshot(s): {count} match IDs"
    )

1 snapshot(s): 802 match IDs
2 snapshot(s): 9962 match IDs
3 snapshot(s): 1616 match IDs
4 snapshot(s): 9 match IDs


In [ ]:
# Define fixed data types for home_team columns
# This prevents datatype conflicts during concatenation

home_team_schema = {
    "match_id": pl.Int64,
    "name": pl.String,
    "slug": pl.String,
    "gender": pl.String,
    "user_count": pl.Int64,
    "residence": pl.String,
    "birthplace": pl.String,
    "height": pl.Float64,
    "weight": pl.Int64,
    "plays": pl.String,
    "turned_pro": pl.Int64,
    "current_prize": pl.Int64,
    "total_prize": pl.Int64,
    "player_id": pl.Int64,
    "current_rank": pl.Int64,
    "name_code": pl.String,
    "country": pl.String,
    "full_name": pl.String
}

In [14]:
# Test schema conversion on one parquet file

test_df = pl.read_parquet(
    home_team_files[0]
)


test_df = test_df.cast(
    home_team_schema,
    strict=False
)


print(test_df.schema)

Schema([('match_id', Int64), ('name', String), ('slug', String), ('gender', String), ('user_count', Int64), ('residence', String), ('birthplace', String), ('height', Float64), ('weight', Int64), ('plays', String), ('turned_pro', Int64), ('current_prize', Int64), ('total_prize', Int64), ('player_id', Int64), ('current_rank', Int64), ('name_code', String), ('country', String), ('full_name', String)])


In [20]:
# Read all home_team parquet files
# Add snapshot date and standardize column types

home_team_frames = []


for file in home_team_files:

    # Read parquet file
    df = pl.read_parquet(file)


    # Extract snapshot date from folder name
    # Example: extracted/20240201/raw_match_parquet/file.parquet
    # Result: 2024-02-01

    snapshot_date = file.parent.parent.name


    # Add snapshot date column

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.strptime(pl.Date, "%Y%m%d")
        .alias("snapshot_date")
    )


    # Apply fixed data types

    df = df.cast(
        home_team_schema,
        strict=False
    )


    home_team_frames.append(df)


print(
    "Number of processed files:",
    len(home_team_frames)
)

Number of processed files: 25610


In [19]:
# Combine all home_team snapshots into one dataframe
# No duplicate removal is performed because snapshots represent different observations

home_team_snapshot = pl.concat(
    home_team_frames,
    how="vertical_relaxed"
)


print(
    "Final shape after concatenation:",
    home_team_snapshot.shape
)

Final shape after concatenation: (25610, 19)


In [17]:
# Check final dataframe schema after concatenation
# This verifies that all columns have consistent data types

home_team_snapshot.schema

Schema([('match_id', Int64),
        ('name', String),
        ('slug', String),
        ('gender', String),
        ('user_count', Int64),
        ('residence', String),
        ('birthplace', String),
        ('height', Float64),
        ('weight', Int64),
        ('plays', String),
        ('turned_pro', Int64),
        ('current_prize', Int64),
        ('total_prize', Int64),
        ('player_id', Int64),
        ('current_rank', Int64),
        ('name_code', String),
        ('country', String),
        ('full_name', String),
        ('snapshot_date', Date)])

In [21]:
# Check missing values in each column

home_team_snapshot.null_count()

match_id,name,slug,gender,user_count,residence,birthplace,height,weight,plays,turned_pro,current_prize,total_prize,player_id,current_rank,name_code,country,full_name,snapshot_date
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,35,0,18423,10836,11287,18578,13126,20806,130,69,0,273,0,9,0,0


In [22]:
# Sort matches by match_id
# Then order snapshots chronologically by snapshot_date

home_team_snapshot = home_team_snapshot.sort(
    [
        "match_id",
        "snapshot_date"
    ]
)

In [ ]:
# Display first rows after sorting

home_team_snapshot.head(10)

In [25]:
# Save processed home_team snapshot dataset

processed_path = Path(DATA_ROOT / "Data")

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

home_team_snapshot.write_parquet(
    processed_path / "home_team.parquet"
)

In [ ]:
# Check the saved parquet file by reading sample rows

# Path of the saved processed dataset
saved_file = processed_path / "home_team.parquet"


# Read the saved parquet file
check_df = pl.read_parquet(saved_file)


# Show dataset information
print("Dataset shape:")
print(check_df.shape)


# Display first 20 rows
print("\nFirst 20 rows:")
check_df.head(20)

In [28]:
# Check important columns in processed dataset

required_columns = [
    "match_id",
    "player_id",
    "user_count",
    "snapshot_date"
]


for column in required_columns:
    if column in check_df.columns:
        print(column, "✓ exists")
    else:
        print(column, "✗ missing")

match_id ✓ exists
player_id ✓ exists
user_count ✓ exists
snapshot_date ✓ exists
